# PyTorch Loss Functions and Optimizers

**PyTorch Interview Preparation · Notebook 05 · Estimated study time: 2–3 hours**

The loop is: **identify task → output shape → target format → loss → optimizer → training order → validate**.


## 1. Setup

All examples are reproducible, CPU-safe, and require only PyTorch.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
print("PyTorch version:",torch.__version__)


## 2. What Is a Loss Function?

A loss function measures how wrong predictions are relative to targets. Training uses gradients of this usually scalar error to improve parameters.

```text
input → model → prediction → loss(prediction,target) → error to minimize
```


## 3. Regression vs Classification — 🔥 Interview Essential

| Task | Typical raw output | Common loss |
|---|---|---|
| Regression | continuous value | `MSELoss` or `L1Loss` |
| Binary classification | one logit/sample | `BCEWithLogitsLoss` |
| Multiclass classification | one logit/class | `CrossEntropyLoss` |
| Multilabel classification | one logit/label | `BCEWithLogitsLoss` |


## 4. `MSELoss` — 🔥 Interview Essential

MSE is `mean((prediction-target)²)`, so large errors receive disproportionately more penalty.


In [ ]:
pred=torch.tensor([2.,4.,6.])
target=torch.tensor([1.,5.,7.])
manual=((pred-target)**2).mean()
loss=nn.MSELoss()(pred,target)
print("manual:",manual,"module:",loss)


### 🟢 Easy — Calculate MSE

Calculate mean-squared error between `pred` and `target` with `nn.MSELoss`.


In [ ]:
# Exercise: Calculate mean-squared error between `pred` and `target` with `nn.MSELoss`.
pred=torch.tensor([1.,3.,7.])
target=torch.tensor([2.,3.,5.])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer.ndim==0, "Default reduction should produce a scalar"
    assert torch.allclose(answer,torch.tensor(5/3)), "Square errors, then average"
    print("✅ Correct!")


### MSE shape safety

Predictions and targets should usually match, e.g. `(32,1)` and `(32,1)`. `(32,1)` versus `(32,)` may broadcast into an unintended `(32,32)` comparison and can emit a warning.


### 🟢 Easy — Match regression target shape

Reshape `target` to match prediction before computing MSE; store reshaped target in `fixed_target` and loss in `answer`.


In [ ]:
# Exercise: Reshape `target` to match prediction before computing MSE; store reshaped target in `fixed_target` and loss in `answer`.
pred=torch.randn(8,1)
target=torch.randn(8)
fixed_target=None
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if fixed_target is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert fixed_target.shape==pred.shape, "Match prediction shape exactly"
    assert answer.ndim==0 and torch.allclose(answer,nn.MSELoss()(pred,fixed_target)), "Compute MSE after fixing shape"
    print("✅ Correct!")


## 5. `L1Loss`

L1 is mean absolute error. MSE squares errors and is more sensitive to outliers; L1 grows linearly and is less sensitive to very large errors.


In [ ]:
pred=torch.tensor([0.,2.,10.]); target=torch.tensor([0.,4.,0.])
print("MSE:",nn.MSELoss()(pred,target),"L1:",nn.L1Loss()(pred,target))


### 🟢 Easy — Calculate L1

Compute mean absolute error.


In [ ]:
# Exercise: Compute mean absolute error.
pred=torch.tensor([1.,5.,9.])
target=torch.tensor([2.,3.,9.])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert torch.allclose(answer,torch.tensor(1.)), "Take absolute errors and average"
    print("✅ Correct!")


## 6. Binary Classification Losses — 🔥 Interview Essential

A common binary model returns one raw logit per sample: `(batch,1)` or consistently `(batch,)`. Large negative logits imply probability near 0; zero maps to 0.5; large positive logits imply probability near 1.


In [ ]:
logits=torch.tensor([[-2.],[0.],[3.]])
print(torch.sigmoid(logits))


## 7. `BCELoss`

`BCELoss` expects **probabilities** between 0 and 1, not raw logits.


In [ ]:
probs=torch.tensor([[.9],[.2],[.7]])
targets=torch.tensor([[1.],[0.],[1.]])
print(nn.BCELoss()(probs,targets))


## 8. `BCEWithLogitsLoss` — 🔥 Interview Essential

It combines sigmoid and binary cross entropy in a numerically stable operation. During training pass raw logits directly.

```text
correct: logits → BCEWithLogitsLoss
wrong:   logits → sigmoid → BCEWithLogitsLoss  # sigmoid effectively applied twice
```


In [ ]:
logits=torch.tensor([[2.],[-1.],[.5]])
targets=torch.tensor([[1.],[0.],[1.]])
loss=nn.BCEWithLogitsLoss()(logits,targets)
print(loss)


### 🟢 Easy — Binary loss

Create the preferred binary-classification loss as `loss_fn`.


In [ ]:
# Exercise: Create the preferred binary-classification loss as `loss_fn`.
loss_fn=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if loss_fn is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(loss_fn,nn.BCEWithLogitsLoss), "Use the stable loss that accepts raw logits"
    print("✅ Correct!")


### 🟢 Easy — Binary predictions

Convert logits to probabilities and threshold at 0.5; store both tensors.


In [ ]:
# Exercise: Convert logits to probabilities and threshold at 0.5; store both tensors.
logits=torch.tensor([[-2.],[.2],[3.]])
probs=None
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if probs is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert torch.allclose(probs,torch.sigmoid(logits)), "Apply sigmoid for inference"
    assert answer.dtype==torch.int64 and torch.equal(answer,torch.tensor([[0],[1],[1]])), "Threshold then convert to integer"
    print("✅ Correct!")


Training: `logits → BCEWithLogitsLoss`. Inference: `logits → sigmoid → threshold`.


## 9. Multiclass Classification — 🔥 Interview Essential

For one class out of many, logits shape is `(batch,classes)`. Class-index targets have shape `(batch,)`, dtype `torch.long`, and values `0 <= target < classes`.


## 10. `CrossEntropyLoss` — 🔥 Interview Essential

Pass raw logits. CrossEntropyLoss combines log-softmax and negative log likelihood stably; do not manually softmax first.


In [ ]:
logits=torch.tensor([[2.,1.,.1],[.5,2.5,.3],[.1,.2,3.]])
targets=torch.tensor([0,1,2],dtype=torch.long)
loss=nn.CrossEntropyLoss()(logits,targets)
print(loss,"predictions:",logits.argmax(dim=1))


### 🟢 Easy — Multiclass loss

Create the standard multiclass class-index loss as `loss_fn`.


In [ ]:
# Exercise: Create the standard multiclass class-index loss as `loss_fn`.
loss_fn=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if loss_fn is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(loss_fn,nn.CrossEntropyLoss), "Use CrossEntropyLoss"
    print("✅ Correct!")


### 🟢 Easy — Multiclass accuracy

Calculate scalar accuracy from logits and class-index targets.


In [ ]:
# Exercise: Calculate scalar accuracy from logits and class-index targets.
logits=torch.tensor([[3.,1.,0.],[0.,2.,1.],[1.,2.,4.],[5.,1.,0.]])
targets=torch.tensor([0,1,1,0])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer.ndim==0 and torch.allclose(answer,torch.tensor(.75)), "Argmax classes, compare, convert to float, average"
    print("✅ Correct!")


## 11. Target Shapes and Dtypes — 🔥 Interview Essential

| Setup | Output | Target | Target dtype |
|---|---|---|---|
| regression | `(B,1)` | `(B,1)` | float |
| binary | `(B,1)` | `(B,1)` | float |
| multiclass | `(B,C)` | `(B,)` | long/int64 |
| multilabel | `(B,L)` | `(B,L)` | float |

Binary vs multiclass high-priority rule: no manual sigmoid/softmax before the corresponding logits loss.


### 🟢 Easy — Fix class targets

Convert `targets` to the normal dtype for class-index CrossEntropyLoss.


In [ ]:
# Exercise: Convert `targets` to the normal dtype for class-index CrossEntropyLoss.
targets=torch.tensor([0.,2.,1.])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer.dtype==torch.long and torch.equal(answer,torch.tensor([0,2,1])), "Use integer class indices"
    print("✅ Correct!")


### Multilabel classification

Multiclass means exactly one class; multilabel means several labels may be true. Multilabel commonly uses `(batch,num_labels)` raw logits, same-shaped float targets such as `[1,0,1,0]`, and `BCEWithLogitsLoss`.


## 12. Loss Reduction

- `mean`: average individual losses into a scalar
- `sum`: add them into a scalar
- `none`: return unreduced element/sample losses for masking, weighting, or inspection


In [ ]:
pred=torch.tensor([1.,2.,3.]); target=torch.tensor([0.,2.,5.])
for reduction in ["mean","sum","none"]:
    result=nn.MSELoss(reduction=reduction)(pred,target)
    print(reduction,result,result.shape)


### 🟢 Easy — Unreduced loss

Return individual squared errors without reduction.


In [ ]:
# Exercise: Return individual squared errors without reduction.
pred=torch.tensor([1.,3.,7.])
target=torch.tensor([0.,3.,5.])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer.shape==(3,) and torch.equal(answer,torch.tensor([1.,0.,4.])), "Use reduction='none'"
    print("✅ Correct!")


## 13. Class Weights

For multiclass imbalance, `CrossEntropyLoss(weight=weights)` increases contribution from selected target classes. It is a tool, not a universal imbalance solution. For binary/multilabel imbalance, `BCEWithLogitsLoss(pos_weight=...)` can increase positive-example contribution.


In [ ]:
weights=torch.tensor([1.,2.,4.])
weighted_ce=nn.CrossEntropyLoss(weight=weights)
pos_weight=torch.tensor([3.])
weighted_bce=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(weighted_ce,weighted_bce)


### 🟡 Medium — Weighted CE

Create CrossEntropyLoss with class weights `[1,2,5]`.


In [ ]:
# Exercise: Create CrossEntropyLoss with class weights `[1,2,5]`.
weights=torch.tensor([1.,2.,5.])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.CrossEntropyLoss), "Create weighted CrossEntropyLoss"
    assert torch.equal(answer.weight,weights), "Pass weights to the loss"
    print("✅ Correct!")


### Task-to-Loss Challenge — Fifteen mappings

For each, name output, target, loss, and inference conversion.

1. House-price regression
2. Cat vs dog, one logit
3. One of 100 image classes
4. Any subset of dog/car/person/tree
5. Predict temperature
6. Spam/not spam
7. Language among 12
8. Five independent medical findings
9. Forecast 24 values
10. Positive/negative review
11. Digit 0–9
12. Multi-hot music genres
13. Bounding-box coordinate regression
14. One intent among 20
15. Churn probability

<details><summary>Show mappings</summary>

1. (B,1) float; target (B,1) float; **MSELoss or L1Loss**; continuous output
2. (B,1) logits; target (B,1) float; **BCEWithLogitsLoss**; sigmoid + threshold
3. (B,100) logits; target (B,) long; **CrossEntropyLoss**; argmax dim=1
4. (B,4) logits; target (B,4) float; **BCEWithLogitsLoss**; sigmoid + per-label threshold
5. (B,1) float; target (B,1) float; **MSELoss/L1Loss**; continuous output
6. (B,1) logits; target (B,1) float; **BCEWithLogitsLoss**; sigmoid + threshold
7. (B,12) logits; target (B,) long; **CrossEntropyLoss**; argmax
8. (B,5) logits; target (B,5) float; **BCEWithLogitsLoss**; five thresholds
9. (B,24) float; target (B,24) float; **MSELoss/L1Loss**; continuous vector
10. (B,1) logits; target (B,1) float; **BCEWithLogitsLoss**; sigmoid
11. (B,10) logits; target (B,) long; **CrossEntropyLoss**; argmax
12. (B,L) logits; target (B,L) float; **BCEWithLogitsLoss**; sigmoid thresholds
13. (B,4) float; target (B,4) float; **L1/MSE (simplified)**; continuous coordinates
14. (B,20) logits; target (B,) long; **CrossEntropyLoss**; argmax
15. (B,1) logits; target (B,1) float; **BCEWithLogitsLoss**; sigmoid + threshold

</details>


## 14. What Is an Optimizer?

An optimizer uses parameter gradients to update parameters toward lower loss.

```text
loss.backward() → parameter.grad → optimizer.step() → changed parameters
backward computes gradients; step updates parameters
```


## 15. Learning Rate — 🔥 Interview Essential

Learning rate controls update size. Too small may be slow; too large may oscillate or diverge. Values such as SGD `1e-2`/`1e-1` and Adam `1e-3` are only heuristics, not universal rules.


## 16. SGD

Plain SGD approximates `new = old - lr × gradient`.


In [ ]:
model=nn.Linear(3,1)
sgd=optim.SGD(model.parameters(),lr=.01)
print(sgd)


### 🟢 Easy — Create SGD

Create SGD for `model` with learning rate 0.01.


In [ ]:
# Exercise: Create SGD for `model` with learning rate 0.01.
model=nn.Linear(2,1)
optimizer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if optimizer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(optimizer,optim.SGD), "Use optim.SGD"
    assert optimizer.param_groups[0]['lr']==.01, "Check learning rate"
    assert {id(p) for p in model.parameters()}=={id(p) for p in optimizer.param_groups[0]['params']}, "Optimize this model's parameters"
    print("✅ Correct!")


## 17. Momentum

SGD momentum maintains a running direction from past gradients, smoothing noisy updates and accelerating consistent directions.


In [ ]:
model=nn.Linear(3,1)
optimizer=optim.SGD(model.parameters(),lr=.01,momentum=.9)
print(optimizer.param_groups[0]['momentum'])


### 🟢 Easy — SGD momentum

Create SGD with lr 0.02 and momentum 0.9.


In [ ]:
# Exercise: Create SGD with lr 0.02 and momentum 0.9.
model=nn.Linear(2,1)
optimizer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if optimizer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(optimizer,optim.SGD), "Use SGD"
    g=optimizer.param_groups[0]
    assert g['lr']==.02 and g['momentum']==.9, "Check both hyperparameters"
    print("✅ Correct!")


## 18. Adam — 🔥 Interview Essential

Adam keeps running estimates of gradient statistics to adapt updates per parameter. It is a convenient starting point in many tasks, but not universally superior.


### 🟢 Easy — Create Adam

Create Adam for `model` with lr `1e-3`.


In [ ]:
# Exercise: Create Adam for `model` with lr `1e-3`.
model=nn.Linear(2,1)
optimizer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if optimizer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(optimizer,optim.Adam), "Use optim.Adam"
    assert optimizer.param_groups[0]['lr']==1e-3, "Check learning rate"
    print("✅ Correct!")


## 19. AdamW — 🔥 Interview Essential for modern AI

AdamW uses decoupled weight decay. It is common in transformers and modern deep learning.


### 🟢 Easy — Create AdamW

Create AdamW with lr `1e-3` and weight decay `1e-2`.


In [ ]:
# Exercise: Create AdamW with lr `1e-3` and weight decay `1e-2`.
model=nn.Linear(2,1)
optimizer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if optimizer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(optimizer,optim.AdamW), "Use optim.AdamW"
    g=optimizer.param_groups[0]
    assert g['lr']==1e-3 and g['weight_decay']==1e-2, "Check lr and weight decay"
    print("✅ Correct!")


## 20. Weight Decay — 🔥 Interview Essential

Weight decay discourages unnecessarily large parameter magnitudes and acts as regularization. Learning rate controls update size; weight decay controls magnitude regularization.


## 21. SGD vs Adam vs AdamW — 🔥 Interview Essential

| Optimizer | Main idea | Common use |
|---|---|---|
| SGD | direct gradient updates | tuned/classical vision training |
| SGD + momentum | smoother directional updates | CNNs, large-scale training |
| Adam | adaptive updates | broad deep-learning use |
| AdamW | adaptive + decoupled decay | transformers/modern AI |

Optimizer choice is problem-dependent. If using weight decay with Adam-style training, AdamW is generally preferred over coupled L2-style Adam decay.


## 22. `zero_grad → backward → step` — 🔥 Interview Essential

```python
optimizer.zero_grad()       # clear accumulated gradients
pred = model(X)             # forward
loss = loss_fn(pred,y)      # scalar objective
loss.backward()             # compute gradients
optimizer.step()            # update parameters
```

`step()` does **not** normally clear `.grad`. Clearing comes before the next independent backward; intentional accumulation is an advanced exception.


### 🟡 Medium — One training step

Complete one MSE training step and assign the scalar loss to `answer`.


In [ ]:
# Exercise: Complete one MSE training step and assign the scalar loss to `answer`.
torch.manual_seed(1)
model=nn.Linear(2,1)
optimizer=optim.SGD(model.parameters(),lr=.1)
loss_fn=nn.MSELoss()
X=torch.tensor([[1.,2.],[2.,1.]])
y=torch.tensor([[1.],[1.]])
before=[p.detach().clone() for p in model.parameters()]
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer.ndim==0, "Calculate scalar loss"
    assert all(p.grad is not None for p in model.parameters()), "Run backward"
    assert any(not torch.equal(p.detach(),old) for p,old in zip(model.parameters(),before)), "Run optimizer.step()"
    print("✅ Correct!")


## 23. Optimizer Parameter Groups

Groups allow different LRs/decay for different parameter subsets, such as a pretrained backbone and new classifier.


In [ ]:
class TwoPart(nn.Module):
    def __init__(self):
        super().__init__(); self.backbone=nn.Linear(4,8); self.classifier=nn.Linear(8,2)
    def forward(self,x): return self.classifier(torch.relu(self.backbone(x)))
model=TwoPart()
optimizer=optim.AdamW([
    {'params':model.backbone.parameters(),'lr':1e-4},
    {'params':model.classifier.parameters(),'lr':1e-3,'weight_decay':0.0},
],weight_decay=1e-2)
print([g['lr'] for g in optimizer.param_groups])


### 🟡 Medium — Two learning rates

Create AdamW groups: backbone lr 1e-4, classifier lr 1e-3.


In [ ]:
# Exercise: Create AdamW groups: backbone lr 1e-4, classifier lr 1e-3.
model=TwoPart()
optimizer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if optimizer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(optimizer,optim.AdamW) and len(optimizer.param_groups)==2, "Use two AdamW parameter groups"
    assert [g['lr'] for g in optimizer.param_groups]==[1e-4,1e-3], "Check group learning rates"
    backbone_ids={id(p) for p in model.backbone.parameters()}
    classifier_ids={id(p) for p in model.classifier.parameters()}
    assert {id(p) for p in optimizer.param_groups[0]['params']}==backbone_ids, "The first group should optimize the backbone"
    assert {id(p) for p in optimizer.param_groups[1]['params']}==classifier_ids, "The second group should optimize the classifier"
    print("✅ Correct!")


## 24. Inspecting Optimizer State

`optimizer.state_dict()` contains configuration/parameter groups and internal momentum/adaptive state. Model and optimizer state dictionaries are both used in checkpointing.


In [ ]:
model=nn.Linear(2,1); optimizer=optim.Adam(model.parameters(),lr=1e-3)
loss=model(torch.ones(1,2)).sum(); loss.backward(); optimizer.step()
state=optimizer.state_dict()
print(state.keys(),"state entries:",len(state['state']))


## 25. Changing Learning Rate

Inspect or modify `optimizer.param_groups`. Schedulers come later.


In [ ]:
for group in optimizer.param_groups: print("before:",group['lr'])
for group in optimizer.param_groups: group['lr']=1e-4
for group in optimizer.param_groups: print("after:",group['lr'])
optimizer.zero_grad(set_to_none=True)
print("set_to_none example:",[p.grad for p in model.parameters()])


### 🟢 Easy — Change LR

Change every optimizer group learning rate to `1e-4`; assign the list of new rates to `answer`.


In [ ]:
# Exercise: Change every optimizer group learning rate to `1e-4`; assign the list of new rates to `answer`.
model=nn.Sequential(nn.Linear(2,3),nn.Linear(3,1))
optimizer=optim.SGD([{'params':model[0].parameters(),'lr':.1},{'params':model[1].parameters(),'lr':.01}])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer==[1e-4,1e-4], "Update all groups"
    assert all(g['lr']==1e-4 for g in optimizer.param_groups), "Modify optimizer.param_groups"
    print("✅ Correct!")


## 26. Common Training Patterns

### Minimal regression


In [ ]:
torch.manual_seed(42)
X=torch.randn(100,1); y=3*X+2+.1*torch.randn(100,1)
model=nn.Linear(1,1); loss_fn=nn.MSELoss(); optimizer=optim.SGD(model.parameters(),lr=.05)
for _ in range(150):
    optimizer.zero_grad(); loss=loss_fn(model(X),y); loss.backward(); optimizer.step()
print("weight/bias/loss:",model.weight.item(),model.bias.item(),loss.item())
assert abs(model.weight.item()-3)<.1 and abs(model.bias.item()-2)<.1


### Minimal binary classification


In [ ]:
torch.manual_seed(42)
X=torch.randn(120,2); targets=(X[:,0]+X[:,1]>0).float().unsqueeze(1)
model=nn.Linear(2,1); loss_fn=nn.BCEWithLogitsLoss(); optimizer=optim.Adam(model.parameters(),lr=.05)
for _ in range(80):
    optimizer.zero_grad(); logits=model(X); loss=loss_fn(logits,targets); loss.backward(); optimizer.step()
with torch.no_grad(): accuracy=((torch.sigmoid(model(X))>=.5)==targets.bool()).float().mean()
print("binary loss/accuracy:",loss.item(),accuracy.item())
assert accuracy>.9


### Minimal multiclass mechanics


In [ ]:
torch.manual_seed(42)
X=torch.randn(64,10); targets=torch.randint(0,3,(64,))
model=nn.Linear(10,3); loss_fn=nn.CrossEntropyLoss(); optimizer=optim.Adam(model.parameters(),lr=1e-2)
optimizer.zero_grad(); logits=model(X); loss=loss_fn(logits,targets); loss.backward(); optimizer.step()
print(logits.shape,targets.shape,loss.item())


### Fifteen loss shape/dtype checks

1. logits (32,1), target (32,1) float → ?
2. logits (32,10), target (32,) long → ?
3. pred (64,1), target (64,1) float → ?
4. logits (8,5), target (8,5) float multi-hot → ?
5. pred (16,4), target (16,4) float continuous → ?
6. logits (1,100), target (1,) long → ?
7. logits (20,), target (20,) float → ?
8. logits (20,2), target (20,) long → ?
9. pred (10,24), target (10,24) → ?
10. logits (12,7), target (12,7) float → ?
11. logits (12,7), target (12,) long → ?
12. probabilities (12,1), target float → ?
13. logits (4,3,16,16), class target (4,16,16) long → ?
14. pred (5,), target (5,) float → ?
15. logits (5,1), target (5,) float → ?

<details><summary>Show answers</summary>

1. BCEWithLogitsLoss
2. CrossEntropyLoss
3. MSE/L1
4. BCEWithLogitsLoss
5. MSE/L1
6. CrossEntropyLoss
7. BCEWithLogitsLoss
8. CrossEntropyLoss
9. MSE/L1
10. BCEWithLogitsLoss for multilabel
11. CrossEntropyLoss for multiclass
12. BCELoss, though logits loss preferred
13. CrossEntropyLoss dense classification
14. MSE/L1 regression
15. shape mismatch: reshape target to (5,1)

</details>


## 27. Common Mistakes

1. Softmax before CrossEntropyLoss.
2. Sigmoid before BCEWithLogitsLoss.
3. Float class indices for CrossEntropyLoss.
4. Integer/long binary targets instead of matching floats.
5. Inconsistent binary output/target shapes.
6. Accidental broadcasting between prediction and target.
7. Forgetting `zero_grad()`.
8. Forgetting `backward()`.
9. Forgetting `step()`.
10. Clearing gradients after backward but before step.
11. Optimizer holds parameters from the wrong model instance.
12. Learning rate so large training diverges.
13. Assuming calculating loss updates parameters.
14. Confusing weight decay regularization with learning rate.
15. Using multiclass CE for independent multilabel targets.


## 28. Debugging Exercises

Predict the correction before opening each answer.


### Debug 1

```python
CE targets are `(32,)` floats
```

What is wrong?

<details><summary>Show diagnosis</summary>

Class-index targets should normally be torch.long.

</details>


### Debug 2

```python
Sigmoid logits before BCEWithLogitsLoss
```

What is wrong?

<details><summary>Show diagnosis</summary>

Pass raw logits; otherwise sigmoid is conceptually applied twice.

</details>


### Debug 3

```python
Softmax logits before CrossEntropyLoss
```

What is wrong?

<details><summary>Show diagnosis</summary>

Pass raw logits; CE contains stable log-softmax internally.

</details>


### Debug 4

```python
zero_grad → loss → step
```

What is wrong?

<details><summary>Show diagnosis</summary>

`loss.backward()` is missing before step.

</details>


### Debug 5

```python
backward → zero_grad → step
```

What is wrong?

<details><summary>Show diagnosis</summary>

Gradients are erased before the update; zero before forward/backward.

</details>


### Debug 6

```python
MSE prediction `(32,1)`, target `(32,)`
```

What is wrong?

<details><summary>Show diagnosis</summary>

Shapes broadcast unintentionally; reshape target to `(32,1)`.

</details>


### Debug 7

```python
BCEWithLogits target dtype long
```

What is wrong?

<details><summary>Show diagnosis</summary>

Use floating targets matching logits shape.

</details>


### Debug 8

```python
Multilabel `(B,L)` uses CrossEntropyLoss
```

What is wrong?

<details><summary>Show diagnosis</summary>

Independent labels require BCEWithLogitsLoss and float multi-hot targets.

</details>


### Debug 9

```python
optimizer.step() called but parameters unchanged
```

What is wrong?

<details><summary>Show diagnosis</summary>

Check backward ran, gradients exist, optimizer owns those exact model parameters, and LR is nonzero.

</details>


### Debug 10

```python
Loss computed but no parameter update expected
```

What is wrong?

<details><summary>Show diagnosis</summary>

Loss evaluation alone does not call backward or step.

</details>


### Debug 11

```python
optimizer created, then model re-instantiated
```

What is wrong?

<details><summary>Show diagnosis</summary>

Optimizer still references old parameters; recreate it for the new model.

</details>


### Debug 12

```python
BCELoss receives values outside [0,1]
```

What is wrong?

<details><summary>Show diagnosis</summary>

BCELoss expects probabilities; use sigmoid first or preferably BCEWithLogitsLoss on logits.

</details>


### Debug 13

```python
CE target contains class index equal to num_classes
```

What is wrong?

<details><summary>Show diagnosis</summary>

Valid indices are 0 through C-1.

</details>


### Debug 14

```python
Binary logits `(B,1)`, targets `(B,)`
```

What is wrong?

<details><summary>Show diagnosis</summary>

BCE logits and targets must match shapes; unsqueeze target.

</details>


### Debug 15

```python
Learning rate 100 causes exploding loss
```

What is wrong?

<details><summary>Show diagnosis</summary>

The update size is far too large; lower/tune LR and inspect gradients.

</details>


### Debug 16

```python
Calling optimizer.step before backward
```

What is wrong?

<details><summary>Show diagnosis</summary>

There are no current gradients to apply.

</details>


### Debug 17

```python
Assume step clears gradients
```

What is wrong?

<details><summary>Show diagnosis</summary>

It does not; call zero_grad for the next independent step.

</details>


### Debug 18

```python
Class weight tensor has wrong length
```

What is wrong?

<details><summary>Show diagnosis</summary>

CrossEntropyLoss weight needs one entry per class.

</details>


### Debug 19

```python
loss_fn = nn.MSELoss(reduction='none')  # result treated as a scalar
```

What is wrong?

<details><summary>Show diagnosis</summary>

It returns unreduced losses; explicitly reduce before backward if a scalar objective is desired.

</details>


### Debug 20

```python
Adam weight_decay described as learning rate
```

What is wrong?

<details><summary>Show diagnosis</summary>

Weight decay regularizes parameter magnitude; LR scales updates.

</details>


## 29. Interview Questions

Answer aloud before expanding.


### 1. What does a loss function do?

<details><summary>Show answer</summary>

**Short Interview Answer:** It measures prediction error against targets and provides an objective to minimize.

**Detailed Explanation:** Backward differentiates the scalar loss with respect to parameters.

</details>


### 2. Regression loss?

<details><summary>Show answer</summary>

**Short Interview Answer:** MSELoss or L1Loss, depending on desired error sensitivity.

**Detailed Explanation:** MSE emphasizes large errors; L1 is more robust to outliers.

</details>


### 3. What is MSELoss?

<details><summary>Show answer</summary>

**Short Interview Answer:** Mean squared difference between prediction and target.

**Detailed Explanation:** Squaring makes it differentiable and penalizes large residuals strongly.

</details>


### 4. MSE vs L1?

<details><summary>Show answer</summary>

**Short Interview Answer:** MSE squares errors; L1 uses absolute errors.

**Detailed Explanation:** MSE is more outlier-sensitive; L1 grows linearly.

</details>


### 5. What is a logit?

<details><summary>Show answer</summary>

**Short Interview Answer:** A raw unnormalized model score.

**Detailed Explanation:** Logits are inputs to stable combined classification losses.

</details>


### 6. What is BCEWithLogitsLoss?

<details><summary>Show answer</summary>

**Short Interview Answer:** Stable sigmoid plus binary cross entropy on raw logits.

**Detailed Explanation:** It avoids numerical issues from separately materializing probabilities.

</details>


### 7. Why prefer it over sigmoid + BCE?

<details><summary>Show answer</summary>

**Short Interview Answer:** The fused formulation is numerically more stable.

**Detailed Explanation:** It combines algebra to avoid extreme-probability log issues.

</details>


### 8. BCE target dtype?

<details><summary>Show answer</summary>

**Short Interview Answer:** Usually floating point, matching logits shape.

**Detailed Explanation:** Values are commonly 0/1, and soft labels can also be floats.

</details>


### 9. Convert binary logits to predictions?

<details><summary>Show answer</summary>

**Short Interview Answer:** Apply sigmoid, then threshold, commonly at 0.5.

**Detailed Explanation:** Threshold choice can be tuned to metric/cost requirements.

</details>


### 10. What is CrossEntropyLoss for?

<details><summary>Show answer</summary>

**Short Interview Answer:** Single-label multiclass classification from raw logits.

**Detailed Explanation:** It expects `(B,C)` logits and usually `(B,)` long class indices.

</details>


### 11. Multiclass logits shape?

<details><summary>Show answer</summary>

**Short Interview Answer:** `(batch_size,num_classes)` for ordinary classification.

**Detailed Explanation:** Dense tasks may include additional spatial dimensions.

</details>


### 12. Class-index target shape?

<details><summary>Show answer</summary>

**Short Interview Answer:** Usually `(batch_size,)`.

**Detailed Explanation:** One integer identifies the correct class for each sample.

</details>


### 13. CE target dtype?

<details><summary>Show answer</summary>

**Short Interview Answer:** torch.long/int64 for class indices.

**Detailed Explanation:** Targets are indices, not probability values in the usual case.

</details>


### 14. Why no softmax before CE?

<details><summary>Show answer</summary>

**Short Interview Answer:** CE expects logits and applies stable log-softmax internally.

**Detailed Explanation:** Pre-softmax changes the intended input and reduces numerical stability.

</details>


### 15. Multiclass vs multilabel?

<details><summary>Show answer</summary>

**Short Interview Answer:** Multiclass chooses one class; multilabel allows multiple independent positives.

**Detailed Explanation:** Multiclass typically uses CE; multilabel uses BCEWithLogitsLoss.

</details>


### 16. Multilabel loss?

<details><summary>Show answer</summary>

**Short Interview Answer:** BCEWithLogitsLoss on one logit per label.

**Detailed Explanation:** Targets are same-shaped float multi-hot vectors.

</details>


### 17. What does reduction none do?

<details><summary>Show answer</summary>

**Short Interview Answer:** Returns individual unreduced losses.

**Detailed Explanation:** Useful for masks, custom weights, and inspection.

</details>


### 18. How can class weighting help?

<details><summary>Show answer</summary>

**Short Interview Answer:** It makes selected target classes contribute more to loss.

**Detailed Explanation:** It can address imbalance but must be tuned/evaluated.

</details>


### 19. What does an optimizer do?

<details><summary>Show answer</summary>

**Short Interview Answer:** It updates parameters using their gradients.

**Detailed Explanation:** Backward computes gradients; optimizer step applies an update rule.

</details>


### 20. What is learning rate?

<details><summary>Show answer</summary>

**Short Interview Answer:** A hyperparameter controlling update size.

**Detailed Explanation:** Too high may diverge; too low may learn very slowly.

</details>


### 21. If LR too high?

<details><summary>Show answer</summary>

**Short Interview Answer:** Training may oscillate, become unstable, or diverge.

**Detailed Explanation:** Loss may spike and parameters can overshoot useful regions.

</details>


### 22. If LR too low?

<details><summary>Show answer</summary>

**Short Interview Answer:** Progress may be extremely slow or appear stalled.

**Detailed Explanation:** More iterations may be needed and optimization can settle poorly.

</details>


### 23. What is SGD?

<details><summary>Show answer</summary>

**Short Interview Answer:** A direct gradient-based optimizer.

**Detailed Explanation:** It updates parameters opposite gradients, often with momentum.

</details>


### 24. What does momentum do?

<details><summary>Show answer</summary>

**Short Interview Answer:** Accumulates a running update direction.

**Detailed Explanation:** It smooths noise and accelerates consistent gradient directions.

</details>


### 25. What is Adam?

<details><summary>Show answer</summary>

**Short Interview Answer:** An adaptive optimizer using running gradient statistics.

**Detailed Explanation:** It adapts effective updates per parameter and often converges conveniently.

</details>


### 26. What is AdamW?

<details><summary>Show answer</summary>

**Short Interview Answer:** Adam-style optimization with decoupled weight decay.

**Detailed Explanation:** It is widely used in transformers and modern training.

</details>


### 27. Adam vs AdamW?

<details><summary>Show answer</summary>

**Short Interview Answer:** AdamW decouples weight decay from Adam's adaptive gradient update.

**Detailed Explanation:** When using decay, AdamW is generally the cleaner formulation.

</details>


### 28. What is weight decay?

<details><summary>Show answer</summary>

**Short Interview Answer:** Regularization discouraging large parameter magnitudes.

**Detailed Explanation:** It differs from LR, which controls update scale.

</details>


### 29. Weight decay vs LR?

<details><summary>Show answer</summary>

**Short Interview Answer:** Decay regularizes magnitude; LR sets step size.

**Detailed Explanation:** They affect training differently and are tuned separately.

</details>


### 30. Why zero_grad?

<details><summary>Show answer</summary>

**Short Interview Answer:** PyTorch accumulates gradients by default.

**Detailed Explanation:** Clearing prevents unintentional mixing across independent steps.

</details>


### 31. What does backward do?

<details><summary>Show answer</summary>

**Short Interview Answer:** Computes and accumulates gradients.

**Detailed Explanation:** It does not update parameters.

</details>


### 32. What does step do?

<details><summary>Show answer</summary>

**Short Interview Answer:** Updates optimizer-managed parameters from gradients.

**Detailed Explanation:** It follows backward in a standard step.

</details>


### 33. Does step clear gradients?

<details><summary>Show answer</summary>

**Short Interview Answer:** No.

**Detailed Explanation:** Call zero_grad separately before the next independent backward.

</details>


### 34. What are parameter groups?

<details><summary>Show answer</summary>

**Short Interview Answer:** Subsets of parameters with different optimizer settings.

**Detailed Explanation:** They enable per-layer learning rates or decay.

</details>


### 35. Why different layer LRs?

<details><summary>Show answer</summary>

**Short Interview Answer:** A pretrained backbone may need smaller updates than a new head.

**Detailed Explanation:** Discriminative rates protect learned features while training new components faster.

</details>


## 30. Knowledge Check Quiz

1. Regression default? A CE B MSE C BCE logits D softmax
2. BCEWithLogits expects: A probabilities B raw logits C argmax D long logits
3. CE expects multiclass logits: A `(B,C)` B `(C,)` only C `(B,1)` D scalar
4. CE class targets: A `(B,C)` float B `(B,)` long C `(B,1)` float D bool
5. Apply softmax before CE? A yes B normally no C only CPU D after backward
6. Binary targets usually: A float matching logits B long classes always C strings D none
7. Binary inference: A argmin B sigmoid+threshold C softmax dim0 D MSE
8. Multiclass inference: A threshold each B argmax classes C sigmoid only D L1
9. Multilabel common loss: A CE B BCEWithLogits C MSE only D NLL on one class
10. reduction none returns: A scalar mean B unreduced losses C optimizer D logits
11. MSE emphasizes: A no errors B large errors C dtype D labels
12. L1 uses: A absolute errors B squared errors C logits D class weights
13. Optimizer does: A calculate loss B update params C create targets D softmax
14. backward does: A update params B compute gradients C zero grads D choose LR
15. Correct order: A step/backward/zero B zero/forward/loss/backward/step C backward/zero/step D loss/step/backward
16. step clears grads? A yes B no C Adam only D SGD only
17. LR controls: A class count B update size C dtype D output shape
18. Momentum: A deletes history B running direction C loss reduction D labels
19. Adam is: A adaptive optimizer B loss C activation D dataset
20. AdamW adds: A softmax B decoupled weight decay C labels D pooling
21. Weight decay is: A update size only B magnitude regularization C target dtype D batch size
22. Parameter groups allow: A different settings B different losses only C no params D GPU only
23. optimizer state_dict contains: A only model weights B optimizer config/state C targets D source code
24. set_to_none: A resets grads to None B freezes model C changes LR D computes loss
25. CE target class range: A 1..C B 0..C-1 C any float D negative only
26. BCELoss expects: A probabilities B raw logits C class indices D embeddings
27. pos_weight is for: A positive BCE contribution B LR C Conv stride D CE classes only
28. Class weights length for CE: A batch size B number of classes C one always D epochs
29. Optimizer after model replacement: A still correct B references old params C auto-updates D freezes
30. Random-label final challenge goal: A perfect accuracy B mechanics correctness C GPU D memorization only

<details><summary>Show answer key</summary>

1 B · 2 B · 3 A · 4 B · 5 B · 6 A · 7 B · 8 B · 9 B · 10 B · 11 B · 12 A · 13 B · 14 B · 15 B · 16 B · 17 B · 18 B · 19 A · 20 B · 21 B · 22 A · 23 B · 24 A · 25 B · 26 A · 27 A · 28 B · 29 B · 30 B

</details>


## 31. Write From Memory


### 🟢 Easy — MSE

Create MSELoss.


In [ ]:
# Exercise: Create MSELoss.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.MSELoss), "Use nn.MSELoss"
    print("✅ Correct!")


### 🟢 Easy — L1

Create L1Loss.


In [ ]:
# Exercise: Create L1Loss.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.L1Loss), "Use nn.L1Loss"
    print("✅ Correct!")


### 🟢 Easy — CE

Create CrossEntropyLoss.


In [ ]:
# Exercise: Create CrossEntropyLoss.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.CrossEntropyLoss), "Use CE"
    print("✅ Correct!")


### 🟢 Easy — BCE logits

Create BCEWithLogitsLoss.


In [ ]:
# Exercise: Create BCEWithLogitsLoss.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.BCEWithLogitsLoss), "Use stable logits loss"
    print("✅ Correct!")


### 🟢 Easy — BCELoss

Create BCELoss.


In [ ]:
# Exercise: Create BCELoss.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.BCELoss), "Use probability BCE"
    print("✅ Correct!")


### 🟢 Easy — Binary probabilities

Convert logits to probabilities.


In [ ]:
# Exercise: Convert logits to probabilities.
logits=torch.tensor([-1.,0.,1.])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert torch.allclose(answer,torch.sigmoid(logits)), "Use sigmoid"
    print("✅ Correct!")


### 🟢 Easy — Binary classes

Threshold sigmoid probabilities at .5 to long.


In [ ]:
# Exercise: Threshold sigmoid probabilities at .5 to long.
logits=torch.tensor([-1.,0.,1.])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert torch.equal(answer,torch.tensor([0,1,1])), "Sigmoid then threshold"
    print("✅ Correct!")


### 🟢 Easy — Multiclass predictions

Convert logits to class indices.


In [ ]:
# Exercise: Convert logits to class indices.
logits=torch.tensor([[1.,3.],[4.,2.]])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert torch.equal(answer,torch.tensor([1,0])), "Argmax class dim"
    print("✅ Correct!")


### 🟢 Easy — SGD

Create SGD lr .01.


In [ ]:
# Exercise: Create SGD lr .01.
model=nn.Linear(2,1)
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,optim.SGD) and answer.param_groups[0]['lr']==.01, "Use SGD"
    print("✅ Correct!")


### 🟢 Easy — SGD momentum

Create SGD lr .01 momentum .9.


In [ ]:
# Exercise: Create SGD lr .01 momentum .9.
model=nn.Linear(2,1)
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,optim.SGD) and answer.param_groups[0]['momentum']==.9, "Set momentum"
    print("✅ Correct!")


### 🟢 Easy — Adam

Create Adam lr 1e-3.


In [ ]:
# Exercise: Create Adam lr 1e-3.
model=nn.Linear(2,1)
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,optim.Adam) and answer.param_groups[0]['lr']==1e-3, "Use Adam"
    print("✅ Correct!")


### 🟢 Easy — AdamW

Create AdamW lr 1e-4, decay .01.


In [ ]:
# Exercise: Create AdamW lr 1e-4, decay .01.
model=nn.Linear(2,1)
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,optim.AdamW) and answer.param_groups[0]['lr']==1e-4 and answer.param_groups[0]['weight_decay']==.01, "Check settings"
    print("✅ Correct!")


### 🟢 Easy — Unreduced MSE

Create MSELoss reduction none.


In [ ]:
# Exercise: Create MSELoss reduction none.
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.MSELoss) and answer.reduction=='none', "Set reduction"
    print("✅ Correct!")


### 🟢 Easy — Weighted CE

Create CE weights `[1,2,3]`.


In [ ]:
# Exercise: Create CE weights `[1,2,3]`.
weights=torch.tensor([1.,2.,3.])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.CrossEntropyLoss) and torch.equal(answer.weight,weights), "Pass weights"
    print("✅ Correct!")


### 🟢 Easy — Positive weight

Create BCEWithLogits pos_weight `[4]`.


In [ ]:
# Exercise: Create BCEWithLogits pos_weight `[4]`.
pw=torch.tensor([4.])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert isinstance(answer,nn.BCEWithLogitsLoss) and torch.equal(answer.pos_weight,pw), "Set pos_weight"
    print("✅ Correct!")


### 🟢 Easy — Inspect LRs

Assign optimizer group LRs to a list.


In [ ]:
# Exercise: Assign optimizer group LRs to a list.
model=nn.Linear(2,1)
optimizer=optim.Adam(model.parameters(),lr=.002)
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer==[.002], "Read param_groups"
    print("✅ Correct!")


### 🟢 Easy — Change LR

Change optimizer LR to 1e-4 and assign it.


In [ ]:
# Exercise: Change optimizer LR to 1e-4 and assign it.
model=nn.Linear(2,1)
optimizer=optim.Adam(model.parameters(),lr=.002)
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer==1e-4 and optimizer.param_groups[0]['lr']==1e-4, "Modify group"
    print("✅ Correct!")


### 🟢 Easy — Zero gradients

Zero optimizer gradients with set_to_none; set answer True.


In [ ]:
# Exercise: Zero optimizer gradients with set_to_none; set answer True.
model=nn.Linear(2,1)
optimizer=optim.SGD(model.parameters(),lr=.1)
model(torch.ones(1,2)).sum().backward()
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer is True and all(p.grad is None for p in model.parameters()), "Use set_to_none=True"
    print("✅ Correct!")


### 🟢 Easy — Training step

Perform zero, forward MSE, backward, step; assign loss.


In [ ]:
# Exercise: Perform zero, forward MSE, backward, step; assign loss.
model=nn.Linear(1,1)
optimizer=optim.SGD(model.parameters(),lr=.1)
X=torch.tensor([[1.]])
y=torch.tensor([[2.]])
before=[p.detach().clone() for p in model.parameters()]
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer.ndim==0, "Calculate a scalar MSE loss"
    assert all(p.grad is not None for p in model.parameters()), "Call backward()"
    assert any(not torch.equal(p.detach(),old) for p,old in zip(model.parameters(),before)), "Call optimizer.step() after backward()"
    print("✅ Correct!")


### 🟢 Easy — Accuracy

Calculate multiclass accuracy.


In [ ]:
# Exercise: Calculate multiclass accuracy.
logits=torch.tensor([[3.,1.],[1.,4.]])
targets=torch.tensor([0,1])
answer=None


In [ ]:
# Automatic validation — run after completing the exercise cell.
if answer is None:
    print("✍️ Complete the exercise above, then run this validation cell.")
else:
    assert answer.item()==1., "Argmax, compare, float mean"
    print("✅ Correct!")


### Optimizer Selection Challenge

Choose a reasonable starting optimizer and explain the tradeoff before opening the discussion.

1. A transformer trained with weight decay.
2. A classical CNN recipe with carefully tuned schedules.
3. A small MLP where a convenient adaptive optimizer is desired.
4. A pretrained backbone and a newly initialized classifier head.

<details><summary>Show discussion</summary>

1. **AdamW** is a common choice because it provides adaptive updates with decoupled weight decay.
2. **SGD with momentum** is common when the recipe is carefully tuned.
3. **Adam or AdamW** are reasonable starting points.
4. **AdamW parameter groups** with a smaller backbone LR and larger head LR can be reasonable.

These are starting points, not unique answers; optimizer choice is empirical and problem-dependent.

</details>


## 32. Final Challenge — Complete Training Components

Random labels mean high accuracy is not the goal. Demonstrate the mechanics correctly.


In [ ]:
torch.manual_seed(42)
X=torch.randn(128,10)
y=torch.randint(0,3,(128,))
model=nn.Sequential(nn.Linear(10,32),nn.ReLU(),nn.Linear(32,3))
initial_parameters=[p.detach().clone() for p in model.parameters()]

loss_fn=None
optimizer=None
first_loss=None
final_loss=None
logits=None
pred=None
accuracy=None
steps_completed=0

# Create CrossEntropyLoss and AdamW(lr=1e-3, weight_decay=1e-2).
# Run 20 correct training steps, recording detached first_loss and final_loss.
# Increment steps_completed once per completed optimizer step.
# Then compute logits, pred, and accuracy under torch.no_grad().


In [ ]:
# Automatic validation — run after completing the challenge.
if accuracy is None:
    print("✍️ Complete all final challenge steps, then run this validation cell.")
else:
    assert isinstance(loss_fn,nn.CrossEntropyLoss), "Use CrossEntropyLoss"
    assert isinstance(optimizer,optim.AdamW), "Use AdamW"
    assert len(optimizer.param_groups)==1, "Use one parameter group for this model"
    group=optimizer.param_groups[0]
    assert group['lr']==1e-3 and group['weight_decay']==1e-2, "Check optimizer settings"
    assert {id(p) for p in group['params']}=={id(p) for p in model.parameters()}, "Optimizer must own this model's parameters"
    assert steps_completed==20, "Complete exactly 20 optimizer steps"
    assert first_loss is not None and final_loss is not None, "Record first and final losses"
    assert first_loss.ndim==0 and final_loss.ndim==0, "Losses should be scalar"
    assert not first_loss.requires_grad and not final_loss.requires_grad, "Detach recorded losses from their training graphs"
    assert final_loss.item()<first_loss.item(), "The repeated steps should reduce loss on this fixed batch"
    assert any(not torch.equal(p.detach(),old) for p,old in zip(model.parameters(),initial_parameters)), "Optimizer steps should change parameters"
    with torch.no_grad():
        expected_logits=model(X)
        expected_pred=expected_logits.argmax(dim=1)
        expected_accuracy=(expected_pred==y).float().mean()
    assert logits.shape==(128,3) and torch.allclose(logits,expected_logits), "Report current raw model logits"
    assert pred.shape==(128,) and pred.dtype==torch.long and torch.equal(pred,expected_pred), "Use argmax across classes"
    assert accuracy.ndim==0 and torch.allclose(accuracy,expected_accuracy), "Compute accuracy from predictions and targets"
    assert all(p.grad is not None for p in model.parameters()), "Backward should populate gradients"
    print(f"✅ Correct! first loss={first_loss.item():.4f}, final loss={final_loss.item():.4f}, accuracy={accuracy.item():.3f}")


## 33. Loss and Optimizer Cheat Sheet

```python
# Regression
nn.MSELoss(); nn.L1Loss()

# Binary/multilabel
loss_fn=nn.BCEWithLogitsLoss()
logits=model(X); loss=loss_fn(logits,targets)
pred=(torch.sigmoid(logits)>=.5).long()

# Multiclass
loss_fn=nn.CrossEntropyLoss()
logits=model(X); loss=loss_fn(logits,targets)
pred=logits.argmax(dim=1)

optim.SGD(model.parameters(),lr=.01)
optim.SGD(model.parameters(),lr=.01,momentum=.9)
optim.Adam(model.parameters(),lr=1e-3)
optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-2)

optimizer.zero_grad()
pred=model(X)
loss=loss_fn(pred,y)
loss.backward()
optimizer.step()

for group in optimizer.param_groups: print(group['lr'])
for group in optimizer.param_groups: group['lr']=1e-4
```

```text
Regression: model → continuous prediction → MSE/L1
Binary: model → one raw logit → BCEWithLogitsLoss
Multiclass: model → one raw logit/class → CrossEntropyLoss

zero_grad → forward → loss → backward → gradients → step → updated parameters
```


### Interview quick reference

| Term | Meaning |
|---|---|
| MSELoss | regression, squared errors |
| BCEWithLogitsLoss | binary/multilabel logits |
| CrossEntropyLoss | multiclass logits + class indices |
| SGD | direct gradient optimizer |
| Adam | adaptive optimizer |
| AdamW | Adam with decoupled weight decay |
| learning rate | update size |
| weight decay | magnitude regularization |
| backward | calculate gradients |
| step | update parameters |
| zero_grad | clear accumulated gradients |


## Before Moving to `06_training_loop.ipynb`

- [ ] I can choose losses for regression, binary, multiclass, and multilabel tasks.
- [ ] I understand logits, target shapes, and target dtypes.
- [ ] I know why CE and BCEWithLogits receive raw logits.
- [ ] I understand loss reduction and basic class weighting.
- [ ] I understand learning rate, momentum, and weight decay.
- [ ] I can use SGD, Adam, and AdamW and explain their differences.
- [ ] I can write `zero_grad → forward → loss → backward → step` from memory.
- [ ] I can inspect parameter groups, learning rates, and optimizer state.
- [ ] I can debug common loss and optimizer mistakes.
